# Ortak test kümesinde son karşılaştırma

Kaydedilmiş 20 tahmin dosyasıyla üç baseline ve dört modeli aynı zamanlarda değerlendirir. Eğitim başlatmaz. Model satırları, ortak kümede ayrı hesaplanan beş seed metriğinin ortalamasıdır. MAE/RMSE kWh, R² birimsizdir.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/enerji-temiz-rerun')
REPO_URL = 'https://github.com/yagmurdamlakara/enerji-tuketimi-tahmini.git'
BRANCH = 'main'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, str(REPO)], check=True)
else:
    # Yerel değişiklikleri silmeden güncel kodu al.
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin',
                    f'{BRANCH}:refs/remotes/origin/{BRANCH}'], check=True)
    branch_exists = subprocess.run(
        ['git', '-C', str(REPO), 'show-ref', '--verify', '--quiet', f'refs/heads/{BRANCH}']
    ).returncode == 0
    checkout = ['checkout', BRANCH] if branch_exists else [
        'checkout', '--no-track', '-b', BRANCH, f'origin/{BRANCH}']
    subprocess.run(['git', '-C', str(REPO), *checkout], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', f'origin/{BRANCH}'], check=True)

packages = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'joblib': 'joblib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

print('Commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Veri ve kayıtlı eğitim klasörü

Ham dosya yolunu ve tamamlanmış eğitim klasörünü kontrol edin. Klasörde run_summary.json, model_test_targets.csv ve 20 prediction CSV bulunmalıdır.


In [ ]:
from datetime import datetime

DATA_PATH = Path('/content/drive/MyDrive/household_power_consumption.txt')
RESULTS_DIR = Path('/content/drive/MyDrive/enerji_temiz_model_sonuclari/20261009_160047_094221')
OUTPUT_DIR = Path('/content/drive/MyDrive/enerji_ortak_test_sonuclari') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')

if not DATA_PATH.is_file():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Tek bir ham veri dosyası seçin.')
    name, content = next(iter(uploaded.items()))
    DATA_PATH = Path('/content') / Path(name).name
    DATA_PATH.write_bytes(content)
if not RESULTS_DIR.is_dir():
    raise FileNotFoundError('RESULTS_DIR yolunu tamamlanmış eğitim klasörüne göre ayarlayın.')


In [ ]:
subprocess.run([sys.executable, str(REPO / 'common_test_comparison.py'),
                '--data', str(DATA_PATH), '--results-dir', str(RESULTS_DIR),
                '--output-dir', str(OUTPUT_DIR)], check=True)


In [ ]:
import json
import pandas as pd

audit = json.loads((OUTPUT_DIR / 'comparison_audit.json').read_text())
assert audit['status'] == 'complete'
print('Ortak timestamp sayısı:', audit['common_test_targets'])
display(pd.read_csv(OUTPUT_DIR / 'common_test_comparison.csv'))
print('Sonuç klasörü:', OUTPUT_DIR)
from google.colab import files
files.download(str(OUTPUT_DIR / 'common_test_comparison.csv'))


## Gönderim ve arşiv

Hocaya sonuç CSV dosyasını, ortak timestamp sayısını ve tabloya dayanan 3–4 cümlelik yorumu gönderin. Detaylı ortak tahmin, seed metrik ve denetim dosyaları çıktı klasöründe arşivlenir. .pth, history ve bütün prediction dosyalarını tekrar göndermeyin. Bu görev makale veya FPGA/HLS değişikliği içermez.
